In [1]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
import xgboost as xgb
import time
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import numpy as np

In [11]:
df_weather = pd.read_csv('../data_file/hanoi_weather_history.csv')
df_air = pd.read_csv('../data_file/hanoi_air_quality_history.csv')

print(df_weather.info())
print(df_air.info())


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 24024 entries, 0 to 24023
Data columns (total 12 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   datetime  24024 non-null  object 
 1   temp      24024 non-null  float64
 2   app_temp  24024 non-null  float64
 3   rh        24024 non-null  int64  
 4   wind_spd  24024 non-null  float64
 5   wind_dir  24024 non-null  int64  
 6   pres      24024 non-null  int64  
 7   vis       24017 non-null  float64
 8   clouds    24024 non-null  int64  
 9   precip    24024 non-null  float64
 10  uv        24024 non-null  float64
 11  dewpt     24024 non-null  float64
dtypes: float64(7), int64(4), object(1)
memory usage: 2.2+ MB
None
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 24058 entries, 0 to 24057
Data columns (total 8 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   datetime  24058 non-null  object 
 1   aqi       24058 non-null  int64  
 2   pm25     

#### 1. Dữ liệu thời tiết

DataFrame này chứa 12 cột, mô tả dữ liệu về thời tiết được thu thập.

| Cột          | Mô tả                                                                                       |
|--------------|---------------------------------------------------------------------------------------------|
| `datetime`   | Thời gian ghi dữ liệu, dạng chuỗi (object), chứa thông tin về ngày và giờ.                 |
| `temp`       | Nhiệt độ thực tế (độ C), kiểu dữ liệu `float64`.                                            |
| `app_temp`   | Nhiệt độ cảm nhận (độ C), kiểu dữ liệu `float64`.                                           |
| `rh`         | Độ ẩm tương đối (%) của không khí, kiểu dữ liệu `int64`.                                     |
| `wind_spd`   | Tốc độ gió (m/s), kiểu dữ liệu `float64`.                                                   |
| `wind_dir`   | Hướng gió (độ từ 0 đến 360), kiểu dữ liệu `int64`.                                           |
| `pres`       | Áp suất khí quyển (hPa), kiểu dữ liệu `int64`.                                              |
| `vis`        | Tầm nhìn (m), kiểu dữ liệu `float64`.                                                      |
| `clouds`     | Tỷ lệ mây che phủ (%), kiểu dữ liệu `int64`.                                                |
| `precip`     | Lượng mưa (mm), kiểu dữ liệu `float64`.                                                     |
| `uv`         | Chỉ số UV (độ mạnh của tia cực tím), kiểu dữ liệu `float64`.                                |
| `dewpt`      | Nhiệt độ sương (độ C), kiểu dữ liệu `float64`.                                              |

#### 2. Dữ liệu chất lượng không khí

DataFrame này chứa 8 cột, mô tả các chỉ số chất lượng không khí.

| Cột        | Mô tả                                                                                       |
|------------|---------------------------------------------------------------------------------------------|
| `datetime` | Thời gian ghi dữ liệu, dạng chuỗi (object), chứa thông tin về ngày và giờ.                 |
| `aqi`      | Chỉ số chất lượng không khí (Air Quality Index), kiểu dữ liệu `int64`.                      |
| `pm25`     | Nồng độ bụi mịn PM2.5 (μg/m³), kiểu dữ liệu `float64`.                                      |
| `pm10`     | Nồng độ bụi PM10 (μg/m³), kiểu dữ liệu `float64`.                                          |
| `o3`       | Nồng độ Ozone (μg/m³), kiểu dữ liệu `float64`.                                              |
| `so2`      | Nồng độ Sulfur Dioxide (μg/m³), kiểu dữ liệu `float64`.                                    |
| `no2`      | Nồng độ Nitrogen Dioxide (μg/m³), kiểu dữ liệu `float64`.                                  |
| `co`       | Nồng độ Carbon Monoxide (μg/m³), kiểu dữ liệu `float64`.                                   |


### Merge dữ liệu từ hai DataFrame

Để có thể dự đoán PM2.5 dựa trên các yếu tố thời tiết, chúng ta cần kết hợp dữ liệu từ hai DataFrame lại với nhau theo cột thời gian.

In [12]:
# Kiểm tra format của cột datetime trong cả hai DataFrame
print("Format datetime trong df_weather:")
print(df_weather['datetime'].head())
print("\nFormat datetime trong df_air:")
print(df_air['datetime'].head())

# Kiểm tra kiểu dữ liệu
print(f"\nKiểu dữ liệu datetime trong df_weather: {df_weather['datetime'].dtype}")
print(f"Kiểu dữ liệu datetime trong df_air: {df_air['datetime'].dtype}")

# Kiểm tra số lượng bản ghi
print(f"\nSố bản ghi df_weather: {len(df_weather)}")
print(f"Số bản ghi df_air: {len(df_air)}")

# Kiểm tra khoảng thời gian
print(f"\nKhoảng thời gian df_weather: từ {df_weather['datetime'].min()} đến {df_weather['datetime'].max()}")
print(f"Khoảng thời gian df_air: từ {df_air['datetime'].min()} đến {df_air['datetime'].max()}")

Format datetime trong df_weather:
0    2022-12-31:17
1    2022-12-31:18
2    2022-12-31:19
3    2022-12-31:20
4    2022-12-31:21
Name: datetime, dtype: object

Format datetime trong df_air:
0    2023-01-30:17
1    2023-01-30:16
2    2023-01-30:15
3    2023-01-30:14
4    2023-01-30:13
Name: datetime, dtype: object

Kiểu dữ liệu datetime trong df_weather: object
Kiểu dữ liệu datetime trong df_air: object

Số bản ghi df_weather: 24024
Số bản ghi df_air: 24058

Khoảng thời gian df_weather: từ 2022-12-31:17 đến 2025-10-30:16
Khoảng thời gian df_air: từ 2022-12-31:17 đến 2025-10-30:17


In [18]:
df_weather['datetime'] = pd.to_datetime(df_weather['datetime'], format='%Y-%m-%d:%H')
df_air['datetime'] = pd.to_datetime(df_air['datetime'], format='%Y-%m-%d:%H')

# Check định dạng sau chuyển đổi
print(f"df_weather datetime type: {df_weather['datetime'].dtype}")
print(f"df_air datetime type: {df_air['datetime'].dtype}")

df_weather datetime type: datetime64[ns]
df_air datetime type: datetime64[ns]


In [20]:
merged_df = pd.merge(df_weather, df_air, on='datetime', how='inner')

In [23]:
print("\nThông tin DataFrame sau khi merge:")
print(f"Số bản ghi: {len(merged_df)}")
print(f"Số cột: {len(merged_df.columns)}")
print(f"Kích thước: {merged_df.shape}")

print(f"\nKhoảng thời gian dữ liệu: từ {merged_df['datetime'].min()} đến {merged_df['datetime'].max()}")

# Hiển thị thông tin về DataFrame đã merge
print("\nThông tin chi tiết DataFrame đã merge:")
merged_df.info()
merged_df.head()


Thông tin DataFrame sau khi merge:
Số bản ghi: 24024
Số cột: 19
Kích thước: (24024, 19)

Khoảng thời gian dữ liệu: từ 2022-12-31 17:00:00 đến 2025-10-30 16:00:00

Thông tin chi tiết DataFrame đã merge:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 24024 entries, 0 to 24023
Data columns (total 19 columns):
 #   Column    Non-Null Count  Dtype         
---  ------    --------------  -----         
 0   datetime  24024 non-null  datetime64[ns]
 1   temp      24024 non-null  float64       
 2   app_temp  24024 non-null  float64       
 3   rh        24024 non-null  int64         
 4   wind_spd  24024 non-null  float64       
 5   wind_dir  24024 non-null  int64         
 6   pres      24024 non-null  int64         
 7   vis       24017 non-null  float64       
 8   clouds    24024 non-null  int64         
 9   precip    24024 non-null  float64       
 10  uv        24024 non-null  float64       
 11  dewpt     24024 non-null  float64       
 12  aqi       24024 non-null  int64        

,datetime,temp,app_temp,rh,wind_spd,wind_dir,pres,vis,clouds,precip,uv,dewpt,aqi,pm25,pm10,o3,so2,no2,co
0,2022-12-31 17:00:00,14.8,14.8,73,0.66,310,1024,10.0,87,0.0,0.0,10.0,155,59.0,73.8,55.7,62.3,9.0,224.5
1,2022-12-31 18:00:00,14.6,14.6,75,1.00,360,1023,10.0,87,0.0,0.0,10.2,171,71.0,88.8,56.0,59.0,6.0,206.0
2,2022-12-31 19:00:00,14.3,14.3,79,1.00,345,1023,10.0,83,0.0,0.0,10.7,179,77.0,96.3,54.0,58.0,6.0,203.7
3,2022-12-31 20:00:00,14.1,14.1,82,1.00,335,1023,10.0,79,0.0,0.0,11.0,199,92.0,115.0,52.0,57.0,6.0,201.3
4,2022-12-31 21:00:00,13.8,13.8,86,1.00,320,1022,10.0,75,0.0,0.0,11.5,161,64.0,80.0,50.0,56.0,6.0,199.0
